# ✈️ AI Travel Planning Agent — Azure AI Foundry

In this notebook you will build a single **Travel Advisor Agent** that combines **four different AI tools** into one intelligent assistant.

| Tool | What it does in this demo |
|------|---------------------------|
| 📄 **File Search** | Searches our destination guide `.txt` files for package details, prices, and itineraries |
| 🐍 **Code Interpreter** | Writes and runs Python code to generate a budget comparison chart |
| 🌐 **Bing Grounding** | Searches the live web for current travel advisories and visa info |
| ⚙️ **Function Calling** | Calls external APIs for live weather and currency exchange rates |

## Architecture
```
User question
      │
      ▼
┌──────────────────────────────────────────────────┐
│           Travel Advisor Agent                    │
│  (single agent — Azure AI Foundry)                │
│                                                  │
│  📄 File Search  │ 🐍 Code Interp │ 🌐 Bing │ ⚙️ Functions │
└──────────────────────────────────────────────────┘
```

The agent automatically picks the right tool (or combination of tools) based on the question.

## Step 1 — Install Packages

In [ ]:
! pip install "azure-ai-agents>=1.2.0b3" azure-ai-projects requests --quiet

## Step 2 — Imports

In [ ]:
import os
import time
import json
import requests
from pathlib import Path

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import (
    BingGroundingTool,
    FileSearchTool,
    CodeInterpreterTool,
    FunctionTool,
    FilePurpose,
)
from IPython.display import Image, display

print("✅ All imports successful")

## Step 3 — Load Credentials & Initialize Clients

We load credentials from `cred.json`. Make sure your file contains:
```json
{
  "PROJECT_ENDPOINT": "https://...",
  "MODEL_DEPLOYMENT_NAME": "gpt-4o",
  "BING_CONNECTION_NAME": "your-bing-connection-name"
}
```

In [ ]:
def find_cred_json(start_path):
    """Walk up the directory tree until cred.json is found."""
    current = Path(start_path)
    while current != current.parent:
        cred_file = current / "cred.json"
        if cred_file.exists():
            return str(cred_file)
        current = current.parent
    return None

cred_file = find_cred_json(os.getcwd())
print(f"Found cred.json at: {cred_file}")

with open(cred_file, "r") as f:
    config = json.load(f)

PROJECT_ENDPOINT = config["PROJECT_ENDPOINT"]
MODEL_NAME       = config.get("MODEL_DEPLOYMENT_NAME", "gpt-4o")
BING_CONN_NAME   = config.get("BING_CONNECTION_NAME")

# AIProjectClient  — used for listing connections (Bing)
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential()
)

# AgentsClient — used for ALL agent operations
agents_client = AgentsClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential()
)

print(f"✅ Clients ready")
print(f"   Model          : {MODEL_NAME}")
print(f"   Bing connection: {BING_CONN_NAME}")

## Step 4 — Upload Destination Files → Vector Store (File Search)

We upload the 4 destination `.txt` files into a **vector store**.
The File Search tool uses this store to answer questions about packages, prices, and itineraries.

In [ ]:
DESTINATIONS_DIR = Path("destinations")
destination_files = sorted(DESTINATIONS_DIR.glob("*.txt"))
print(f"Found {len(destination_files)} files: {[f.name for f in destination_files]}\n")

# Upload each .txt file to the agents service
uploaded_file_ids = []
for fp in destination_files:
    upl = agents_client.files.upload_and_poll(
        file_path=str(fp),
        purpose=FilePurpose.AGENTS
    )
    uploaded_file_ids.append(upl.id)
    print(f"  Uploaded: {fp.name}  →  File ID: {upl.id}")

# Group them into a single vector store
vector_store = agents_client.vector_stores.create_and_poll(
    file_ids=uploaded_file_ids,
    name="Travel Destinations"
)
print(f"\n✅ Vector store ready: '{vector_store.name}'  (ID: {vector_store.id})")

## Step 5 — Define Function Tools (Function Calling)

We define two Python functions the agent can call when it needs live data:
- `fetch_weather(location)` — current weather from OpenWeatherMap API
- `get_exchange_rate(from_currency, to_currency, amount)` — live currency rate

> **Note:** Replace `OPENWEATHER_API_KEY` with your free key from [openweathermap.org](https://openweathermap.org/api)

In [ ]:
OPENWEATHER_API_KEY = "your-api-key"  # ← replace with your key


def fetch_weather(location: str) -> str:
    """Get current weather for a given city."""
    try:
        url = "http://api.openweathermap.org/data/2.5/weather"
        params = {"q": location, "appid": OPENWEATHER_API_KEY, "units": "metric"}
        resp = requests.get(url, params=params)
        resp.raise_for_status()
        d = resp.json()
        return json.dumps({
            "location": location,
            "temperature_c": d["main"]["temp"],
            "condition": d["weather"][0]["description"].capitalize(),
            "humidity_pct": d["main"]["humidity"],
            "wind_kmh": round(d["wind"]["speed"] * 3.6, 1)
        })
    except Exception as e:
        return json.dumps({"error": str(e), "location": location})


def get_exchange_rate(from_currency: str, to_currency: str, amount: float = 1.0) -> str:
    """Get live exchange rate between two currencies."""
    try:
        url = f"https://api.exchangerate-api.com/v4/latest/{from_currency.upper()}"
        resp = requests.get(url)
        resp.raise_for_status()
        rate = resp.json()["rates"].get(to_currency.upper())
        if not rate:
            return json.dumps({"error": f"Currency '{to_currency}' not found"})
        return json.dumps({
            "from": from_currency.upper(),
            "to": to_currency.upper(),
            "rate": rate,
            "amount": amount,
            "converted": round(amount * rate, 2)
        })
    except Exception as e:
        return json.dumps({"error": str(e)})


# Wrap both functions in a FunctionTool so the agent knows their signatures
function_tool = FunctionTool(functions={fetch_weather, get_exchange_rate})
print(f"✅ FunctionTool ready with {len(function_tool.definitions)} function(s)")

## Step 6 — Create the Agent with All 4 Tools

This is the key step. We create **one agent** and attach all four tools at once.
The agent will automatically decide which tool to use based on the question.

In [ ]:
# --- Bing: look up the connection ID by name ---
bing_connection_id = None
for conn in project_client.connections.list():
    if conn["name"] == BING_CONN_NAME:
        bing_connection_id = conn["id"]
        print(f"✅ Bing connection found: {BING_CONN_NAME}")
        break

if not bing_connection_id:
    print(f"⚠️  Bing connection '{BING_CONN_NAME}' not found — Bing queries will not work.")

# --- Build the four tool objects ---
bing_tool  = BingGroundingTool(connection_id=bing_connection_id)
file_tool  = FileSearchTool(vector_store_ids=[vector_store.id])
code_tool  = CodeInterpreterTool()
# function_tool was already created in the previous cell

# --- Combine all tool definitions into one list ---
all_tool_defs = (
    bing_tool.definitions +
    file_tool.definitions +
    code_tool.definitions +
    function_tool.definitions
)

# --- Tool resources: use file_tool.resources directly (contains the vector store ID) ---
tool_resources = file_tool.resources

# --- Create the agent ---
agent = agents_client.create_agent(
    model=MODEL_NAME,
    name="travel-advisor-agent",
    instructions="""
You are a helpful AI travel planning assistant. You help users plan trips by:
1. Searching destination guide files for package details, prices, and itineraries.
2. Searching the web (Bing) for live travel advisories, visa requirements, and current news.
3. Writing and running Python code to calculate budgets and generate charts.
4. Calling functions to get live weather forecasts and currency exchange rates.

Always be friendly and cite your sources when possible.
""",
    tools=all_tool_defs,
    tool_resources=tool_resources,
)

print(f"\n✅ Travel agent created!")
print(f"   Agent ID  : {agent.id}")
print(f"   Tools     : {[t['type'] for t in all_tool_defs]}")

## Step 7 — Create Thread & ask() Helper

We create one conversation thread and a reusable `ask()` function.

The `ask()` function:
1. Sends the question to the thread
2. Starts a run and polls until it finishes
3. If the agent needs to call a function (`requires_action`), it handles that automatically
4. Prints the agent's response (text + any generated chart image)

In [ ]:
# One thread for the whole conversation
thread = agents_client.threads.create()
print(f"✅ Thread created: {thread.id}")


def ask(question):
    print(f"\n{'='*65}")
    print(f"YOU: {question}")
    print('='*65)

    # Add the user's message to the thread
    agents_client.messages.create(
        thread_id=thread.id,
        role="user",
        content=question
    )

    # Start a run
    run = agents_client.runs.create(
        thread_id=thread.id,
        agent_id=agent.id
    )

    # Poll until done — handle function calls when status is requires_action
    while run.status in ["queued", "in_progress", "requires_action"]:
        time.sleep(1)
        run = agents_client.runs.get(thread_id=thread.id, run_id=run.id)

        if run.status == "requires_action":
            tool_outputs = []
            for tc in run.required_action.submit_tool_outputs.tool_calls:
                fn_name = tc.function.name
                args    = json.loads(tc.function.arguments)
                print(f"  ⚙️  Calling: {fn_name}({args})")

                if fn_name == "fetch_weather":
                    result = fetch_weather(args["location"])
                elif fn_name == "get_exchange_rate":
                    result = get_exchange_rate(
                        args["from_currency"],
                        args["to_currency"],
                        args.get("amount", 1.0)
                    )
                else:
                    result = json.dumps({"error": f"Unknown function: {fn_name}"})

                tool_outputs.append({"tool_call_id": tc.id, "output": result})

            agents_client.runs.submit_tool_outputs(
                thread_id=thread.id,
                run_id=run.id,
                tool_outputs=tool_outputs
            )

    print(f"\nStatus: {run.status}")

    # Print the latest assistant message (messages are returned newest-first)
    messages = agents_client.messages.list(thread_id=thread.id)
    for msg in messages:
        if msg.role == "assistant":
            for part in msg.content:
                if hasattr(part, "text"):
                    print(f"\nAGENT:\n{part.text.value}")
                elif hasattr(part, "image_file"):
                    # Code Interpreter produced a chart — save and display it
                    fid      = part.image_file.file_id
                    img_path = f"chart_{fid}.png"
                    stream   = agents_client.files.get_content(fid)
                    with open(img_path, "wb") as f:
                        for chunk in stream:
                            f.write(chunk)
                    print(f"\n📊 Chart saved → {img_path}")
                    display(Image(filename=img_path))
            break  # only show the most recent assistant reply


print("\n✅ ask() helper ready — let's start asking questions!")

---
## Demo 1 — File Search 📄

**Question:** What's included in the Andaman package?

**Tool used:** File Search → agent searches `Andaman.txt` in the vector store and returns the answer with a file citation.

In [ ]:
ask("What is included in the Andaman Islands package? What are the important notes I should know before booking?")

---
## Demo 2 — Code Interpreter 🐍

**Question:** Show a bar chart comparing all 4 package costs.

**Tool used:** Code Interpreter → agent writes Python + matplotlib code, executes it, and returns a chart image.

In [ ]:
ask(
    "Here are 4 travel packages with their approximate costs in USD: "
    "Andaman ($385), Europe Grand Tour ($3500), Paris ($4000), Switzerland ($3100). "
    "Create a bar chart comparing these package costs. "
    "Use a different color for each bar and add the price as a label on top of each bar."
)

---
## Demo 3 — Bing Grounding 🌐

**Question:** Latest travel advisories for Switzerland.

**Tool used:** Bing Grounding → agent searches the live web and returns current information with web citations.

In [ ]:
ask("What are the current travel advisories and visa requirements for Indian citizens visiting Switzerland in 2025?")

---
## Demo 4 — Function Calling ⚙️

**Question:** Weather in Port Blair + USD to INR rate today.

**Tool used:** Function Calling → agent calls `fetch_weather("Port Blair")` and `get_exchange_rate("USD", "INR")` automatically.

In [ ]:
ask("What is the current weather in Port Blair, Andaman? Also, what is today's USD to INR exchange rate?")

---
## Bonus — Multi-Tool Query 🔀

This single question makes the agent use **three tools together**:

1. **File Search** → fetches Paris package details and total cost from `Paris.txt`
2. **Function Calling** → gets live USD → INR exchange rate
3. **Bing Grounding** → searches the web for current France travel alerts

Watch how the agent orchestrates all three automatically.

In [ ]:
ask(
    "I am planning a Paris trip. "
    "From the destination guide, what is the total package cost and what is included? "
    "Convert the total cost to INR at today's exchange rate. "
    "Also, are there any current travel alerts for France I should be aware of?"
)

---
## Cleanup 🧹

Delete the agent, vector store, and uploaded files when you are done.

In [ ]:
# Delete the agent
agents_client.delete_agent(agent.id)
print(f"🗑️  Deleted agent: {agent.id}")

# Delete the vector store
agents_client.vector_stores.delete(vector_store.id)
print(f"🗑️  Deleted vector store: {vector_store.id}")

# Delete the uploaded files
for fid in uploaded_file_ids:
    agents_client.files.delete(fid)
print(f"🗑️  Deleted {len(uploaded_file_ids)} uploaded destination files")

print("\n✅ Cleanup complete!")

---
## Summary — What You Built

In this notebook you created a **single AI agent** that can:

| # | Demo | Tool Used | Output |
|---|------|-----------|--------|
| 1 | Package details from documents | File Search | Text answer + file citation |
| 2 | Budget bar chart | Code Interpreter | Generated PNG chart |
| 3 | Live travel advisories | Bing Grounding | Web-sourced answer + URL citations |
| 4 | Weather + exchange rate | Function Calling | Live API data |
| Bonus | All-in-one travel query | File Search + Function + Bing | Combined answer |

The key insight: **one agent, four tools, zero manual routing** — the model decides which tool to use.